In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os
import json


TARGET_WELLS = ['15/9-F-12', '15/9-F-14']

print(f'\t Loading data ')
df = pd.read_parquet('../data/processed/cleaned.parquet')
df['DATEPRD'] = pd.to_datetime(df['DATEPRD'])
df = df.set_index('DATEPRD').sort_index()
df['OIL_RATE_NORM'] = df.groupby('NPD_WELL_BORE_NAME')['OIL_RATE_NORM'].transform(
    lambda x :x.clip(upper= x.quantile(0.99))
)
print(f'Loaded  {len(df):,} rows')

print(f'Resampling to monthly')
AGG_RULES = {
    'OIL_RATE_NORM': 'mean',
    'BORE_OIL_VAL' : 'mean',
    'AVG_WHP_P':      'mean',
    'AVG_DOWNHOLE_PRESSURE':    'mean',
    'WCT':          'mean',
    'GOR' :         'median',
    'DRAWDOWN':     'mean',
    'DAYS_ON_PROD' : 'max',
    'ON_STREAM_HRS' : 'sum',
}

monthly_well_data = {}

for well_name,well_df in df.groupby('NPD_WELL_BORE_NAME'):
    if well_name not in TARGET_WELLS:
        print(f'Skipping wells : {well_name}')
        continue
        
    cols_available = { k: v for k, v in AGG_RULES.items() if k in well_df.columns}

    monthly = well_df[list(cols_available.keys())].resample('ME').agg(cols_available).dropna(
        subset=['OIL_RATE_NORM']
    )

    monthly_well_data[well_name] = monthly
    print(f'{well_name} : {len(monthly)} months')


	 Loading data 
Loaded  8,020 rows
Resampling to monthly
Skipping wells : 15/9-F-1 C
Skipping wells : 15/9-F-11


AttributeError: 'DataFrame' object has no attribute 'colums'